# Selecting Scorio Math answers with `scorio.aggregate`

This notebook chooses one answer from each pool in a 10-question AIME 2026 window. The
main analysis projects answer and verifier columns from the Bucket. A final section reads
one full low-reasoning pool to reproduce confidence signals from its top-20 distributions.
The first analysis opens 10 remote Parquet files, so runtime depends on network latency.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

from scorio import agg, eval

BUCKET_ROOT = "hf://buckets/harimo/scorio-math"


def pool_path(model, task, question_id):
    return f"{BUCKET_ROOT}/data/{model}/{task}/q{question_id:02d}.parquet"


def read_pools(model, task, question_ids, columns, max_workers=2):
    """Read selected columns from question files, preserving question order."""
    paths = [pool_path(model, task, q) for q in question_ids]

    def read_one(path):
        return pq.read_table(path, columns=columns)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        tables = list(executor.map(read_one, paths))
    return pa.concat_tables(tables).to_pandas()

model_name = "gpt-oss-20b_low"
task = "aime_2026"
question_count = 10
question_ids = range(question_count)
columns = [
    "data_id", "seed", "extracted_answer", "evalscope_is_correct", "cv3b_abc_A",
    "llmv_problem_understanding_expected", "llmv_reasoning_validity_expected",
    "llmv_conclusion_support_expected",
]

rows = read_pools(model_name, task, question_ids, columns).sort_values(["data_id", "seed"])
M, N = question_count, 80

answers = rows.extracted_answer.to_numpy().reshape(M, N).astype(object)
answers[answers == "NotFound"] = None
cv3b = rows.cv3b_abc_A.to_numpy().reshape(M, N)

llmv_expected = (
    rows.llmv_problem_understanding_expected
    + rows.llmv_reasoning_validity_expected
    + rows.llmv_conclusion_support_expected
).to_numpy().reshape(M, N) / 3
llmv = (llmv_expected - 1) / 19

accepted = [
    set(group.loc[group.evalscope_is_correct.astype(bool), "extracted_answer"])
    for _, group in rows.groupby("data_id", sort=True)
]

print(answers.shape, cv3b.shape, llmv.shape)


(10, 80) (10, 80) (10, 80)


## One question and eight candidates


In [2]:
question_id = 0
pool = answers[question_id, :8]
scores = cv3b[question_id, :8]

print("candidates    ", list(pool))
print("P(correct)    ", scores.round(3))
print("first sample  ", pool[0])
print("majority_vote ", agg.majority_vote(pool))
print("best_of_n     ", agg.best_of_n(pool, scores))
print("accepted      ", accepted[question_id])


candidates     ['277', '277', '277', '277', '277', '277', '277', '277']
P(correct)     [1. 1. 1. 1. 1. 1. 1. 1.]
first sample   277
majority_vote  277
best_of_n      277
accepted       {'277'}


## Select one answer for every question

Selection methods return labels. The helper checks each label against the rule-based grades
for that question and reports average accuracy with its Bayesian uncertainty.


In [3]:
def accuracy(selected):
    hit = np.array([[int(answer in accepted[i])] for i, answer in enumerate(selected)])
    mu, sigma = eval.avg(hit)
    return {"accuracy": round(float(mu), 3), "sigma": round(float(sigma), 3)}


n = 8
A, V, L = answers[:, :n], cv3b[:, :n], llmv[:, :n]
results = {
    "first sample": accuracy(A[:, 0]),
    "majority_vote": accuracy(agg.majority_vote(A)),
    "weighted_majority_vote (CV3B)": accuracy(agg.weighted_majority_vote(A, V)),
    "best_of_n (CV3B)": accuracy(agg.best_of_n(A, V)),
    "best_of_n (reference-free verifier)": accuracy(agg.best_of_n(A, L)),
}
display(pd.DataFrame(results).T.sort_values("accuracy", ascending=False))


,accuracy,sigma
weighted_majority_vote (CV3B),0.9,0.224
best_of_n (reference-free verifier),0.9,0.224
best_of_n (CV3B),0.9,0.224
majority_vote,0.8,0.224
first sample,0.6,0.224


## Sample-budget sweep


In [4]:
budgets = [1, 2, 4, 8, 16, 32, 80]
sweep = pd.DataFrame({
    n: {
        "majority_vote": accuracy(agg.majority_vote(answers[:, :n]))["accuracy"],
        "weighted (CV3B)": accuracy(
            agg.weighted_majority_vote(answers[:, :n], cv3b[:, :n])
        )["accuracy"],
        "best_of_n (CV3B)": accuracy(
            agg.best_of_n(answers[:, :n], cv3b[:, :n])
        )["accuracy"],
    }
    for n in budgets
}).T
sweep.index.name = "samples"
display(sweep)


,majority_vote,weighted (CV3B),best_of_n (CV3B)
samples,,,
1,0.6,0.6,0.6
2,0.6,0.7,0.7
4,0.7,0.8,0.8
8,0.8,0.9,0.9
16,0.8,0.9,0.9
32,0.9,1.0,1.0
80,0.8,1.0,1.0


## Confidence from the top-20 distributions

This section reads one full low-reasoning pool. Scorio expects numeric log-probability rows,
so the candidate structs are converted before computing each signal.


In [5]:
full_pool = pq.read_table(pool_path(model_name, task, 0)).to_pylist()


def topk_logprobs(record):
    return [[item["logprob"] for item in position]
            for position in record["tokens"]["completion_topk_logprobs_list"]]


signals = []
for record in full_pool[:3]:
    topk = topk_logprobs(record)
    signals.append({
        "seed": record["seed"],
        "self_certainty": agg.self_certainty(topk),
        "deepconf": agg.deepconf_confidence(topk),
        "negative_entropy": -agg.token_entropy(topk),
        "max_probability": agg.max_softmax_probability(topk),
        "logprob_margin": agg.logprob_margin(topk),
    })
display(pd.DataFrame(signals).round(4))


,seed,self_certainty,deepconf,negative_entropy,max_probability,logprob_margin
0,0,12.5181,15.5141,-0.2020,0.9263,9.1336
1,1,12.3201,15.3164,-0.2314,0.9145,8.6585
2,2,12.8419,15.8378,-0.1917,0.9303,9.5222


The [aggregation reference](https://github.com/mohsenhariri/scorio/blob/main/scorio/aggregate/README.md)
maps published methods to their confidence signal and selection rule.
